In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
pip install -U transformers peft accelerate safetensors!python convert_hf_to_gguf.py \
  /content/drive/MyDrive/merged-fp16 \
  --outfile model-fp16.gguf \
  --outtype f16

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 106.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "/content/drive/MyDrive/adapters"
MERGED_PATH = "/content/drive/MyDrive/merged-fp16"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="cpu"
)

model = PeftModel.from_pretrained(model, ADAPTER_PATH)


model = model.merge_and_unload()


model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ LoRA merged and FP16 model saved


In [12]:
!git clone https://github.com/ggerganov/llama.cpp

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [13]:
%cd llama.cpp

/content/llama.cpp


In [14]:
!cmake -B build
!cmake --build build --config Release

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [26]:
!pip install -U llama-cpp-python

  Using cached llama_cpp_python-0.3.16.tar.gz (50.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4503276 sha256=5140ce384f6a408f57e52875a7029d95c36477cda2ebe92bc387a30833733a40
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [35]:
%cd /content/llama.cpp

/content/llama.cpp


In [36]:
!python convert_hf_to_gguf.py \
  /content/drive/MyDrive/merged-fp16 \
  --outfile model-fp16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.wei

In [38]:
!build/bin/llama-quantize model-fp16.gguf model-q8_0.gguf q8_0

main: build = 8185 (2afcdb977)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-fp16.gguf' to 'model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                   

In [42]:
!build/bin/llama-quantize model-fp16.gguf model-q4_0.gguf q4_0

main: build = 8185 (2afcdb977)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-fp16.gguf' to 'model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                   

In [44]:
!ls -lh model-fp16.gguf model-q8_0.gguf model-q4_0.gguf

-rw-r--r-- 1 root root 2.1G Mar  2 11:37 model-fp16.gguf
-rw-r--r-- 1 root root 608M Mar  2 11:41 model-q4_0.gguf
-rw-r--r-- 1 root root 1.1G Mar  2 11:40 model-q8_0.gguf


In [45]:
!build/bin/llama-cli -m model-fp16.gguf -p "Analyze this employee profile briefly." -n 128


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8185-2afcdb977
model      : model-fp16.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Analyze this employee profile briefly.

|-\|/-\|/-\ Profile: Josephine, 42, Human Resources, Dept: Human Resources, Salary: $4,714, Satisfaction: 3/5, Attrition: No

Key HR metrics: 5yr tenure, OverTime: No, Distance: 7mi, JobLevel: 4, Outcome: No

Responsibilities: Human Resources, Dept: Human Resources, Salary: $4,714, Satisfaction: 3/5

Profile summary: Josephine works in Human Re

In [46]:
!build/bin/llama-cli -m model-q8_0.gguf -p "Analyze this employee profile briefly." -n 128


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8185-2afcdb977
model      : model-q8_0.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Analyze this employee profile briefly.

|-\|/-\|/-\|/-\|/-\ Profile: Maria, 26, Human Resources, Dept: Human Resources, Salary: $4,547, Satisfaction: 1/5, Attrition: No

Responsibilities: Manages human resources for a small company. Responsible for hiring, training, and managing employees. Reports to HR director. 9yr exp.

Educa

In [47]:
!build/bin/llama-cli -m model-q4_0.gguf -p "Analyze this employee profile briefly." -n 128


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8185-2afcdb977
model      : model-q4_0.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Analyze this employee profile briefly.

|-\|/-\|/- Input:
Employee #165: Age 28, Male, Human Resources, Dept: Human Resources, Salary: $2,500, Satisfaction: 2/5, Attrition: No
<|user|>
Hey, that's a pretty good profile. Can you add some more details about the company?

[ Prompt: 26.4 t/s | Generation: 10.7 t/s ]

> 
/content/llama.cpp/bu

In [48]:
!mkdir -p /content/drive/MyDrive/quantized/model-int8
!mkdir -p /content/drive/MyDrive/quantized/model-int4

In [49]:
!mv model-q8_0.gguf /content/drive/MyDrive/quantized/model-int8/
!mv model-q4_0.gguf /content/drive/MyDrive/quantized/model-int4/

In [50]:
!cp /content/drive/MyDrive/quantized/model-int8/model-q8_0.gguf \
     /content/drive/MyDrive/quantized/model.gguf